# Mechanics sandbox

Try game mechanics before anyone writes a contract or a screen.

- **Same math as the real thing.** Every number here comes from `backend/sdk/seedlib.mjs` (kept bit-for-bit equal to `SeedLib.sol` by `backend/test/SeedVectors.t.sol`) and from the frontend's own models in `frontend/src/lib/`. Both are read only.
- **Reproducible.** Round seeds are fixed (`keccak256("sandbox-<i>")`), so the same cell gives the same numbers.
- **Rules still apply** (CLAUDE.md): randomness only from the seed, never block data; a 2% edge on house games (return 0.98 per chip); odds that can be shown honestly before a bet.

**Setup:** Node 18+ and a Python kernel with pandas and matplotlib. The first call runs `npm install` in `sandbox/` by itself.

For results you want to commit and share, put the experiment in `SANDBOX.md` and run `npm run sandbox`. That file is the record. This notebook is the workbench.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sandbox as sb

pd.set_option("display.precision", 3)
%matplotlib inline

## 1. Parity: does the UI compute what the chain will pay?

Frontend math vs the backend toolkit on 500 seeds. Every row should say `match`. If one doesn't, the UI would show a different result than the contract pays.

In [ ]:
sb.parity()

## 2. The frontend's games, as they are right now

Every choice of every game in `frontend/src/lib/games.ts`, played with `resolveBet`, the function the UI uses. `claimedChance` and `claimedPayout` are what the UI shows the player. `mean` is what actually comes back per chip. Red bars fail the 2% edge check.

In [ ]:
games = sb.frontend_games(rounds=10_000, beauty_rounds=200)
house = games[games.kind != "beauty"]
house[["game", "label", "claimedChance", "claimedPayout", "paidRate", "mean", "ci95", "status"]]

In [ ]:
ids = house.game_id.unique()
fig, axes = plt.subplots(1, len(ids), figsize=(4 * len(ids), 3.2), gridspec_kw={"width_ratios": [max(2, (house.game_id == i).sum()) for i in ids]})
for ax, gid in zip(axes, ids):
    g = house[house.game_id == gid]
    sb.plot_returns(g, title=g.game.iloc[0], ax=ax)
    ax.set_ylim(0.6, 1.4)
    ax.tick_params(axis="x", labelrotation=45, labelsize=8)
plt.tight_layout()
plt.show()

## 3. Crash: does the curve really give 2% at every target?

`SeedLib.crashX100` should give `P(crash ≥ T) = 0.98 / T`. The experiment's `resolve` returns the crash point, so we can compare the whole curve with the theory, not just the nine ladder targets.

In [ ]:
CRASH = '''
({
  name: 'Crash point',
  gameId: 3,
  resolve(seed, t) { return t.crashX100(seed) },
  pays(x, choice) { return x >= 200n ? 2 : 0 },   // a 2.00x target, for the table
})
'''
crash = sb.experiment(CRASH, rounds=50_000, keep=True)
points = pd.Series(crash.outcomes, dtype=float) / 100
crash

In [ ]:
targets = [1.1, 1.25, 1.5, 2, 3, 5, 10, 20, 50, 100]
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.4))
a.hist(points.clip(upper=20), bins=200, color="#3b6ea5")
a.set_xlabel("crash point (x), clipped at 20"); a.set_ylabel("rounds"); a.set_title("Distribution")
a.spines[["top", "right"]].set_visible(False)

xs = pd.Series(targets)
b.plot(xs, [(points >= x).mean() for x in xs], "o", label="simulated")
b.plot(xs, 0.98 / xs, "-", color="#555", label="0.98 / target")
b.set_xscale("log"); b.set_yscale("log")
b.set_xlabel("target (x)"); b.set_ylabel("chance to reach it"); b.set_title("P(crash ≥ target)")
b.legend(frameon=False); b.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()
# 2% of rounds crash instantly (below 1.00x, shown as 1.00x). Integer hundredths round another
# ~1% (1.000x to 1.009x) down to 1.00x too, so ~3% show 1.00x. No ladder target is below 1.10x.
print(f"Rounds showing 1.00x: {(points == 1).mean():.2%} (expect about 3%: 2% instant crash + 1% rounded down)")

## 4. Beauty Contest: which fixed guess beats the sim bots?

Not a house game: winners split the pot. A return above 1.00 means a strategy beats the bots, and the bots' entries are minted, so the house pays for it. This sweeps every guess from 0 to 40.

In [ ]:
bc = sb.frontend_games(rounds=1, beauty_rounds=1_000, guesses="0-40")
bc = bc[bc.kind == "beauty"]
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(bc.choice, bc["mean"], color="#3b6ea5")
ax.fill_between(bc.choice, bc["mean"] - bc.ci95, bc["mean"] + bc.ci95, alpha=0.2, color="#3b6ea5")
ax.axhline(1, color="#555", linestyle="--", linewidth=1, label="break even")
ax.set_xlabel("your guess"); ax.set_ylabel("return per entry"); ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.show()
best = bc.loc[bc["mean"].idxmax()]
print(f"Best fixed guess: {int(best.choice)}, return {best['mean']:.2f}x per entry")

## 5. Your experiment

Write the mechanic in JavaScript, in the same format as `SANDBOX.md`. `t` is the backend toolkit: `draw`, `uniform`, `chance`, `weighted`, `coin`, `crashX100`, `subSeed`, `unit` (see `backend/docs/MECHANICS.md`). Return a payout multiplier: 0 means the stake is lost, 1.96 means it pays 1.96x.

Use `resolve` + `pays` when the round is costly to compute: `resolve` runs once per round, and `pays` once per choice.

In [ ]:
WHEEL = '''
({
  name: 'Prize wheel',
  choices: ['spin'],
  resolve(seed, t) {
    return t.weighted(seed, 0, [55, 25, 15, 5])  // slice index
  },
  pays(slice, choice) {
    return [0, 1.6, 2.4, 4.4][slice]             // 0.25*1.6 + 0.15*2.4 + 0.05*4.4 = 0.98
  },
})
'''
wheel = sb.experiment(WHEEL, rounds=20_000, keep=True)
wheel

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1, 3]})
sb.plot_returns(wheel.table, title=wheel.name, ax=a)
sb.plot_convergence(wheel.mults, ax=b)
b.set_title("A session's average swings a lot before it settles at 0.98")
plt.tight_layout()
plt.show()

## 6. Block Walk: fair price at each step

In Block Walk, players trade YES (the walk ends above 0) while it moves. The fair YES price with the marker at `x` and `k` steps left is the chance that a `k`-step walk ends above `-x`. This compares that formula, which the frontend's learning mode would show, with walks simulated from the toolkit.

The simulation fixes all 21 steps from one seed. That's fine for checking prices, but the real game must not: players trade between steps, so each step needs a seed nobody knows yet (see MECHANICS.md).

In [ ]:
WALK = '''
({
  name: 'Block Walk paths',
  expectedReturn: null,
  resolve(seed, t) {
    const path = [0]
    for (let k = 0; k < 21; k++) path.push(path[k] + (t.coin(t.subSeed(seed, k)) ? 1 : -1))
    return path
  },
  pays(path, choice) { return path[21] > 0 ? 1 : 0 },   // value of one YES share, in pots
})
'''
walk = sb.experiment(WALK, rounds=10_000, keep=True)
paths = pd.DataFrame(walk.outcomes)

In [ ]:
from math import comb

def fair_yes(x, k):
    # P(x + sum of k fair ±1 steps > 0)
    return sum(comb(k, u) for u in range(k + 1) if x + 2 * u - k > 0) / 2 ** k

step = 10  # marker position after 10 of 21 steps
df = pd.DataFrame({"x": paths[step], "yes": paths[21] > 0})
g = df.groupby("x").agg(simulated=("yes", "mean"), walks=("yes", "size")).reset_index()
g["fair"] = [fair_yes(x, 21 - step) for x in g.x]
g = g[g.walks >= 50]

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(g.x, g.fair * 100, "-", color="#555", label="fair price (formula)")
ax.plot(g.x, g.simulated * 100, "o", color="#3b6ea5", label="simulated")
ax.set_xlabel(f"marker position after step {step}"); ax.set_ylabel("YES price (chips of 100)")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.show()
g

## Next

- An experiment worth keeping goes into `SANDBOX.md` as a ```` ```js experiment ```` block, then run `npm run sandbox` and commit, so the team can see it.
- When a mechanic is ready, it becomes a contract in `backend/contracts/games/` using the same `SeedLib` calls, and the frontend copies its math. The parity section above then checks that they agree.